# Session 1: how heat flows through a slab
**A. Real-world problem.** The chip inside a laptop gets hot; the metal heatsink and the air
around it stay cooler. Heat always moves from the hot side to the cold side, but how fast? And
what does the temperature look like INSIDE the material between them? Today we answer both
for the simplest case: a flat slab with one hot face and one cold face.

**Plan.** (1) write Fourier's law of conduction, (2) get the temperature at any depth,
(3) compare materials, (4) plot the straight-line temperature profile.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
AREA = 1.0e-4          # m^2: a 1 cm x 1 cm patch, about the size of a small chip

# thermal conductivity k in W/(m K) (typical room-temperature values)
MATERIALS = {
    "copper": 400.0,
    "aluminium": 200.0,
    "steel": 50.0,
    "thermal pad": 5.0,
    "glass": 1.0,
    "still air": 0.026,
}

## Step 1: Fourier's law
Heat flow through a slab of thickness L, area A and conductivity k, with the hot face at
T_hot and the cold face at T_cold (steady state: nothing changes with time):

    Q = k * A * (T_hot - T_cold) / L          (watts)

Read it as a story: a bigger area lets more heat through, a bigger temperature difference
pushes harder, a thicker slab slows the heat down, and a higher conductivity k means the
material is a better heat road. Heat flux q = Q / A is the same thing per square metre (W/m^2).

In [ ]:
def heat_rate(k, thickness_mm, t_hot, t_cold, area=AREA):
    """Heat flow Q in watts through a slab of the given thickness (mm)."""
    return k * area * (t_hot - t_cold) / (thickness_mm * 1e-3)


def heat_flux(k, thickness_mm, t_hot, t_cold):
    """Heat flux q in W/m^2 (heat per second per square metre)."""
    return k * (t_hot - t_cold) / (thickness_mm * 1e-3)

## Step 2: the temperature inside the slab
In steady state the same heat Q must cross every layer, so the temperature falls at a constant
rate: a straight line from T_hot at x = 0 to T_cold at x = L. Notice that k does not appear
in this line. A good conductor and a poor one have the same straight profile; they differ in
how much heat Q they carry.

In [ ]:
def temperature_at(x_mm, thickness_mm, t_hot, t_cold):
    """Temperature (deg C) at depth x_mm from the hot face."""
    return t_hot - (t_hot - t_cold) * np.asarray(x_mm, dtype=float) / thickness_mm

## Step 3: compare materials
Same slab (5 mm thick, 1 cm^2), same temperatures (80 C hot face, 40 C cold face): how much
heat flows in each material?

In [ ]:
def compare_materials(thickness_mm=5.0, t_hot=80.0, t_cold=40.0):
    return {name: heat_rate(k, thickness_mm, t_hot, t_cold) for name, k in MATERIALS.items()}

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    table = compare_materials()
    for name, q in table.items():
        print(f"{name:12s} k = {MATERIALS[name]:7.3f} W/(m K)  ->  Q = {q:9.3f} W")
    print(f"copper carries {table['copper'] / table['thermal pad']:.0f} times more heat than the thermal pad")

    q = heat_rate(5.0, 1.0, 80.0, 50.0)
    print(f"worked example: pad k = 5, 1 mm, 80 C to 50 C -> Q = {q:.1f} W, flux {heat_flux(5.0, 1.0, 80.0, 50.0):.0f} W/m^2")
    print(f"temperature at mid depth = {float(temperature_at(0.5, 1.0, 80.0, 50.0)):.1f} C")

    fig, ax = plt.subplots(figsize=(6, 4))
    for thick in (2.0, 5.0, 10.0):
        x = np.linspace(0, thick, 50)
        ax.plot(x, temperature_at(x, thick, 80.0, 40.0), label=f"L = {thick:.0f} mm")
    ax.set_xlabel("depth from the hot face (mm)")
    ax.set_ylabel("temperature (C)")
    ax.set_title("Straight-line temperature profile, any material")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "temperature_profile.png", dpi=120)

    names = [n for n in table if n != "still air"]
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.bar(names, [table[n] for n in names], color="#2a9d8f")
    ax2.set_yscale("log")
    ax2.set_ylabel("heat flow Q (W), log scale")
    ax2.set_title("Same slab, different material (5 mm, 80 C to 40 C)")
    ax2.grid(alpha=0.3, axis="y")
    fig2.tight_layout()
    fig2.savefig(OUT / "heat_by_material.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint. Next session: what if there are several layers, and the air at the end?

## Coding tasks
1. **Easy.** Run `lesson.py` and change the slab thickness in `heat_rate` from 5 mm to 10 mm. Predict the new copper heat flow first, then check it.
2. **Stretch.** Write `thickness_for(q_watts, k, t_hot, t_cold)` that returns the largest slab thickness in mm that still carries q_watts through 1 cm^2, and test it with Section F answer D.
3. **Explain.** Explain why a thin layer of still air (k about 0.026) between a chip and a heatsink is so harmful, using the numbers from the material comparison.

## Assessment
- **A. Conceptual.** A slab of copper and a slab of glass have the same size and the same face temperatures. Which has the larger temperature drop per millimetre, and which carries more heat? Explain.
- **B. Numerical.** A thermal pad with k = 5 W/(m K) is 0.5 mm thick and covers 1 cm^2, with faces at 70 C and 50 C. Find the heat flow in W and check that the answer is sensible.
- **C. Graph and simulation interpretation.** In the profile graph for L = 2, 5 and 10 mm (80 C to 40 C), what is the temperature at a depth of 2 mm in each slab, and what does that tell you about the slope?
- **D. Design and optimisation challenge.** Design: a pad (k = 5 W/(m K), 1 cm^2) must carry at least 12 W from a chip face at 80 C to a heatsink face at 50 C. Manufacturing says it cannot be thinner than 0.3 mm. Choose a thickness and justify it.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("worked example: thermal pad (k = 5, 1 mm, 80 C to 50 C) carries 15.0 W",
                   abs(L.heat_rate(5.0, 1.0, 80.0, 50.0) - 15.0) < 1e-9))
    checks.append(("flux is 150000 W/m^2 for the worked example",
                   abs(L.heat_flux(5.0, 1.0, 80.0, 50.0) - 150000.0) < 1e-6))
    checks.append(("hand calculation: copper 5 mm, 80 C to 40 C carries 320 W",
                   abs(L.heat_rate(400.0, 5.0, 80.0, 40.0) - 320.0) < 1e-9))
    checks.append(("temperature at mid depth is the average of the faces (65 C)",
                   abs(float(L.temperature_at(0.5, 1.0, 80.0, 50.0)) - 65.0) < 1e-9))
    checks.append(("the profile starts at T_hot and ends at T_cold",
                   abs(float(L.temperature_at(0.0, 5.0, 80.0, 40.0)) - 80.0) < 1e-9
                   and abs(float(L.temperature_at(5.0, 5.0, 80.0, 40.0)) - 40.0) < 1e-9))
    checks.append(("doubling the thickness halves the heat flow",
                   abs(L.heat_rate(50.0, 10.0, 80.0, 40.0) * 2 - L.heat_rate(50.0, 5.0, 80.0, 40.0)) < 1e-9))
    checks.append(("no temperature difference means no heat flow", L.heat_rate(400.0, 5.0, 60.0, 60.0) == 0.0))
    t = L.compare_materials()
    checks.append(("copper carries 80 times more heat than the thermal pad", abs(t["copper"] / t["thermal pad"] - 80.0) < 1e-9))
    x = np.linspace(0, 5, 11)
    checks.append(("the profile is a straight line (constant slope)",
                   np.allclose(np.diff(L.temperature_at(x, 5.0, 80.0, 40.0)), -8.0 * 0.5)))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
